# SecAgg: Secure Aggregation

**Library:** [Flower](https://flower.ai/) SecAgg+ (Apache-2.0)  
**Install:** `pip install flwr`  
**Stage:** During training (mask individual updates from the server)

---

## What SecAgg Does

Without SecAgg, the server sees each client's raw model update — which can leak training data. SecAgg adds **pairwise masks** that cancel when summed, so the server only sees the aggregate.

```
Client 0:  update + mask_01 + mask_02
Client 1:  update - mask_01 + mask_12
Client 2:  update - mask_02 - mask_12
─────────────────────────────────────
Server sum: update_0 + update_1 + update_2  (all masks cancel)
```

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

## Step 1: Pairwise Masking Demo

In [ ]:
import numpy as np
from fl_pets.secagg import mask_parameters, verify_cancellation

# Original model update (4 parameters)
params = [np.array([1.5, -0.3, 2.7, 0.01])]

# Mask for each of 5 clients
num_clients = 5
round_seed = 42

print('Individual masked updates (each looks random):')
for c in range(num_clients):
    masked = mask_parameters(params, client_id=c, num_clients=num_clients, round_seed=round_seed)
    print(f'  Client {c}: {[round(x, 4) for x in masked[0]]}')

# Verify masks cancel when summed
result = verify_cancellation(params, num_clients=num_clients, round_seed=round_seed)
print(f'\nServer aggregate:  {[round(x, 4) for x in result["aggregate"][0]]}')
print(f'Expected (5x sum): {[round(x, 4) for x in result["expected"][0]]}')
print(f'Max error: {result["max_error"]:.2e}')

## Step 2: SecAgg + DP Together

SecAgg and DP are complementary:

| Technique | Protects against | Adds noise? |
|-----------|-----------------|-------------|
| **DP** | Server analysing aggregate trends | Yes (accuracy cost) |
| **SecAgg** | Server seeing individual updates | No (lossless) |
| **DP + SecAgg** | Both threats | Yes (DP noise only) |

In production, use both: SecAgg hides individual updates, DP protects the aggregate.

## Step 3: Dropout Tolerance

Flower's SecAgg+ handles client dropout. If a client fails mid-round, remaining clients can still complete aggregation — the masks for the dropped client are reconstructed from shared secrets.

In [ ]:
# Simulate dropout: client 3 drops out
active_clients = [0, 1, 2, 4]  # client 3 dropped

masked_all = []
for c in active_clients:
    masked = mask_parameters(params, client_id=c, num_clients=num_clients, round_seed=round_seed)
    masked_all.append(masked)

partial_sum = [sum(m[0] for m in masked_all)]
print(f'Partial aggregate (4/5 clients): {[round(x, 4) for x in partial_sum[0]]}')
print(f'Expected (4x original):          {[round(x * 4, 4) for x in params[0]]}')
print('\nNote: with real SecAgg+, Flower reconstructs the dropped mask from shared secrets.')
print('This demo uses deterministic masks so partial sums show the drift.')

## References

- Bonawitz et al. (2017), *Practical Secure Aggregation for Privacy-Preserving Machine Learning*, CCS
- Bell et al. (2020), *Secure Single-Server Aggregation with (Poly)Logarithmic Overhead*, CCS
- [Flower SecAgg+](https://flower.ai/) — Apache-2.0

## Next Steps

- [HE: Encrypted Inference](he-encrypted-inference.ipynb) — compute on encrypted data
- [Tutorial 5](../intermediate/05-secure-aggregation.ipynb) — SecAgg in the full FL training loop